In [45]:
from pathlib import Path

import io
import onnx
import onnxruntime.training.onnxblock as onnxblock
import torch
from onnxsim import simplify

import sys
sys.path.append('..')
from model import Model

In [46]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [47]:
# Create a classifier instance
device = "cpu"
batch_size = 8
num_classes = 102
channels = 3
img_h, img_w = 128, 128

model_name = 'resnet10t' # resnet18, resnet34, mobilenetv2_100, efficientnet_b2, visformer_tiny

model_frozen_layer_num = {
    'resnet18': 12,
    'resnet34': 20,
    'mobilenetv2_100': 21,
    'efficientnet_b2': 27,
    'visformer_tiny': 24,
    'lcnet_100': 19,
    'semnasnet_100': 20,
    'mobilenetv3_large_100': 22,
    'resnet10t': 14,
}
frozen_layer_num = model_frozen_layer_num[model_name]

# artifacts path
artifacts_path = Path(f'../artifacts/{model_name}/artifacts_cpu/frozen_layer_{frozen_layer_num}_classes_{num_classes}')
artifacts_path.mkdir(parents=True, exist_ok=True)

pt_model = Model(model_name=model_name, num_classes=num_classes, channels=channels, frozen_layer_num=frozen_layer_num, img_size=img_h)
print(f'Model trainable parameters: {count_parameters(pt_model)}')

for param in pt_model.named_parameters():
    if 'frozen_features' in param[0] or 'bn' in param[0] or 'downsample.1' in param[0]:
        param[1].requires_grad = False
    else:
        param[1].requires_grad = True

torch.save(pt_model, artifacts_path / f'{model_name}.pth')
pt_model = torch.load(artifacts_path / f'{model_name}.pth')
for param in pt_model.named_parameters():
    if param[1].requires_grad:
        print(param[0].ljust(40), '->\t', param[1].requires_grad)

print(f'Model trainable parameters: {count_parameters(pt_model)}')

2024-09-21 00:32:52,352 timm.models._builder [INFO] - Loading pretrained weights from Hugging Face hub (timm/resnet10t.c3_in1k)
2024-09-21 00:32:52,594 urllib3.connectionpool [DEBUG] - https://huggingface.co:443 "HEAD /timm/resnet10t.c3_in1k/resolve/main/model.safetensors HTTP/11" 302 0
2024-09-21 00:32:52,599 timm.models._hub [INFO] - [timm/resnet10t.c3_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.


Number of layers: 16
Requires grad:
	1.   Conv2d:                   True	->	frozen
	2.   BatchNorm2d:              True	->	frozen
	3.   ReLU:                     False	->	frozen
	4.   Conv2d:                   True	->	frozen
	5.   BatchNorm2d:              True	->	frozen
	6.   ReLU:                     False	->	frozen
	7.   Conv2d:                   True	->	frozen
	8.   BatchNorm2d:              True	->	frozen
	9.   ReLU:                     False	->	frozen
	10.  MaxPool2d:                False	->	frozen
	11.  BasicBlock:               True	->	frozen
	12.  BasicBlock:               True	->	frozen
	13.  BasicBlock:               True	->	frozen
	14.  BasicBlock:               True	->	frozen
	15.  SelectAdaptivePool2d:     False	->	frozen
	16.  Linear:                   True	->	trainable
Number of layers requiring grad: 11
Number of trainable blocks: 1
Model trainable parameters: 4974814
output.weight                            ->	 True
output.bias                              ->	 True
Mo

/tmp/ipykernel_100778/3265814655.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pt_model = torch.load(artifacts_path / f'{model_name}.pth')


### Generating forward-only graph

In [48]:
# Generate a random input.
model_inputs = (torch.randn(batch_size, channels, img_h, img_w, device=device),)

model_outputs = pt_model(*model_inputs)
if isinstance(model_outputs, torch.Tensor):
    model_outputs = [model_outputs]
    
input_names = ["input"]
output_names = ["output"]
dynamic_axes = {"input": {0: "batch_size"}, "output": {0: "batch_size"}}

f = io.BytesIO()
torch.onnx.export(
    pt_model,
    model_inputs,
    f,
    input_names=input_names,
    output_names=output_names,
    opset_version=17,
    do_constant_folding=False,
    # training=torch.onnx.TrainingMode.PRESERVE,
    dynamic_axes=dynamic_axes,
    export_params=True,
    # keep_initializers_as_inputs=False,
)

onnx_model = onnx.load_model_from_string(f.getvalue())
print(f'Simplifying ONNX {model_name} model...')
eval_model, check = simplify(onnx_model)
onnx.save(onnx_model, artifacts_path / f'{model_name}.onnx')

Simplifying ONNX resnet10t model...


In [49]:
# device = "cpu"
# batch_size = 64
# num_classes = 10
# channels = 3
# img_h, img_w = 128, 128
# frozen_layer_num = 12

# model_name = 'resnet18' # resnet18, resnet34, mobilenetv2_100, efficientnet_b2, visformer_tiny

# # artifacts path
# artifacts_path = Path('./test_resnet18')
# artifacts_path.mkdir(parents=True, exist_ok=True)

# onnx_model = onnx.load('./test_resnet18.onnx')

### Generating training graph

Generating artifacts based on forward-only graph and the specified loss function using onnxblock library

In [50]:
# Creating a class with a Loss function
class CustomTrainingBlock(onnxblock.TrainingBlock):
    def __init__(self):
        super(CustomTrainingBlock, self).__init__()
        self.loss = onnxblock.loss.CrossEntropyLoss()

    def build(self, output_name):
        return self.loss(output_name), output_name

In [51]:
# Build the onnx model with loss
training_block = CustomTrainingBlock()
for param in onnx_model.graph.initializer:   
    if 'frozen_features' in param.name or 'bn' in param.name or 'downsample.1' in param.name:
        training_block.requires_grad(param.name, False)
    else:
        print(param.name.ljust(40), '\t--->\tlearnable')
        training_block.requires_grad(param.name, True)

# Building training graph and eval graph
model_params = None
with onnxblock.base(onnx_model):
    build_output = training_block(*[output.name for output in onnx_model.graph.output])
    print('Graph output nodes:', build_output)
    training_model, eval_model = training_block.to_model_proto()
    model_params = training_block.parameters()

# Building the optimizer graph
optimizer_block = onnxblock.optim.AdamW()
with onnxblock.empty_base() as accessor:
    _ = optimizer_block(model_params)
    optimizer_model = optimizer_block.to_model_proto()

2024-09-21 00:32:54,352 root [DEBUG] - Building training block CustomTrainingBlock


2024-09-21 00:32:54,358 root [DEBUG] - Building block: CrossEntropyLoss
2024-09-21 00:32:54,470 root [DEBUG] - Building gradient graph for training block CustomTrainingBlock
2024-09-21 00:32:54,540 root [DEBUG] - The loss output is onnx::loss::26. The gradient graph will be built starting from onnx::loss::26_grad.


output.weight                            	--->	learnable
output.bias                              	--->	learnable


2024-09-21 00:32:54,591 root [DEBUG] - Adding gradient accumulation nodes for training block CustomTrainingBlock
2024-09-21 00:32:54,618 root [DEBUG] - Building forward block AdamW
2024-09-21 00:32:54,627 root [DEBUG] - Building block: AdamWOptimizer


Graph output nodes: ('onnx::loss::26', 'output')


In [52]:
# Saving all the files to use them later for the training.
onnxblock.save_checkpoint(training_block.parameters(), artifacts_path / 'checkpoint')

ir_version = 9
opset_import_version = 17

training_model.ir_version = ir_version
# print('Simplifying ONNX training model...')
# training_model, check = simplify(training_model)
# assert check, "Simplified ONNX training model could not be validated"
onnx.save(training_model, artifacts_path / 'training_model.onnx')

optimizer_model.ir_version = ir_version
optimizer_model.opset_import[1].version = opset_import_version
onnx.save(optimizer_model, artifacts_path / 'optimizer_model.onnx')

eval_model.ir_version = ir_version
# print('Simplifying ONNX eval model...')
# eval_model, check = simplify(eval_model)
# assert check, "Simplified ONNX eval model could not be validated"
onnx.save(eval_model, artifacts_path / 'eval_model.onnx')

print(f'Artifacts saved in {artifacts_path} directory')

Artifacts saved in ../artifacts/resnet10t/artifacts_cpu/frozen_layer_14_classes_102 directory


### Verify model

In [53]:
# from pathlib import Path

# import numpy as np
# from onnxruntime.training.api import CheckpointState, Module, Optimizer

In [54]:
# # artifacts path
# device = 'cpu'
# artifacts_path = Path(f'../artifacts/{model_name}/artifacts_cpu/frozen_layer_{frozen_layer_num}_classes_{num_classes}')

# print('Loading artifacts...')
# # Create checkpoint state
# state = CheckpointState.load_checkpoint(artifacts_path / 'checkpoint')

# # Create module
# print(f'Creating model with {device} device...')
# model = Module(
#     artifacts_path / 'training_model.onnx',
#     state,
#     artifacts_path / 'eval_model.onnx',
#     device=device,
# )

# # Create optimizer
# optimizer = Optimizer(artifacts_path / 'optimizer_model.onnx', model)
# optimizer.set_learning_rate(0.001)

In [55]:
# print(f'Model inputs: {model.input_names()}')
# print(f'Model outputs: {model.output_names()}')

# batch_size = 32
# forward_inputs = [
#     np.ones((batch_size, 3, 128, 128), dtype=np.float32),
# ]
# model.train()
# train_loss, logits = model(*forward_inputs)